In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 06 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 06.C1 — Add chains over F2

Represent a chain by the set of simplices with coefficient 1. Return the sum of two chains.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Symmetric difference keeps items present in exactly one set.

</details>

In [3]:
def add_chains(a,b):
    return set(a)^set(b)

In [4]:
assert add_chains({(0,1),(1,2)},{(1,2),(0,2)})=={(0,1),(0,2)}
assert add_chains({(0,1)},{(0,1)})==set()
print("Repeated simplices cancel over F2")

Repeated simplices cancel over F2


**Why this works and what it does not establish:** This is coefficient arithmetic, not geometric deletion. A simplex appearing twice has coefficient 1+1=0 in F2.

## 06.C2 — Implement binary matrix rank

Implement Gaussian elimination over F2 for a two-dimensional integer array. Return rank; do not call rank_mod2 inside your solution.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Find a pivot, swap it into place, XOR it into lower rows, and count pivots.

</details>

In [5]:
def my_rank_f2(matrix):
    A=(np.asarray(matrix,dtype=int)%2).copy();row=0
    for col in range(A.shape[1]):
        candidates=np.flatnonzero(A[row:,col])
        if len(candidates)==0: continue
        pivot=row+int(candidates[0]);A[[row,pivot]]=A[[pivot,row]]
        for i in range(row+1,A.shape[0]):
            if A[i,col]: A[i]^=A[row]
        row+=1
        if row==A.shape[0]: break
    return row

In [6]:
rng=np.random.default_rng(62)
for shape in [(0,3),(3,0),(3,3),(5,7)]:
    A=rng.integers(0,2,size=shape)
    assert my_rank_f2(A)==rank_mod2(A)
assert my_rank_f2([[1,1,0],[1,0,1],[0,1,1]])==2
print("Binary rank checked on empty and nonempty arrays")

Binary rank checked on empty and nonempty arrays


**Why this works and what it does not establish:** Every row operation preserves the row-space dimension over F2. Zero-row and zero-column matrices arise naturally at the ends of chain complexes.

## 06.C3 — Use the Betti rank formula

Given the number n_k of k-simplices and ranks of boundary maps out of and into C_k, return beta_k. Assume valid chain-complex data.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

First subtract noncycles; then subtract independent cycles that are boundaries.

</details>

In [7]:
def betti_from_ranks(n_k,rank_out,rank_in):
    return n_k-rank_out-rank_in

In [8]:
assert betti_from_ranks(3,2,0)==1
assert betti_from_ranks(3,2,1)==0
print("Triangle outline H1=1; filled triangle H1=0")

Triangle outline H1=1; filled triangle H1=0


**Why this works and what it does not establish:** The formula uses rank-nullity and the fact that the incoming boundary image lies inside the outgoing kernel. Arbitrary unrelated matrices need not satisfy that containment.

## 06.C4 — Compare cycles with boundaries

Return beta_1 for a triangle outline and the filled triangle, in that order, using the supplied finite homology helpers.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The new triangle does not erase its boundary edges; it changes which cycles are trivial in homology.

</details>

In [9]:
def triangle_h1_pair():
    outline=closure([(0,1),(1,2),(0,2)])
    filled=closure([(0,1,2)])
    return betti_numbers(outline,1)[1],betti_numbers(filled,1)[1]

In [10]:
assert triangle_h1_pair()==(1,0)
K=closure([(0,1,2)])
assert np.all((boundary_matrix(K,1)@boundary_matrix(K,2))%2==0)
print("Cycle remains an edge chain, but becomes a boundary when a face enters")

Cycle remains an edge chain, but becomes a boundary when a face enters


**Why this works and what it does not establish:** This distinction prevents the common mistake that a class dies because a loop-shaped set of edges literally disappears.

## 06.C5 — Count loops across an image threshold

Return H1 Betti numbers of the declared closed-pixel model at each requested threshold.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Rebuild the subcomplex at each threshold; the complete rectangular image is filled at 1.

</details>

In [11]:
def image_betti1(image,thresholds):
    f=pixel_filtration(1-np.asarray(image,dtype=float)/16)
    return [betti_numbers([s for s,v in f if v<=t],1)[1] for t in thresholds]

In [12]:
image,sample_id=digit_example(8)
values=image_betti1(image,[0,.25,.5,.75,1])
assert all(v>=0 for v in values) and values[-1]==0
print("Training image",sample_id,"H1 counts:",values)

Training image 8 H1 counts: [0, 1, 2, 2, 0]


**Why this works and what it does not establish:** The counts can increase and decrease. They are properties of the filtered pixel object, not a guaranteed match to the number of holes a reader sees in the printed digit.